# 🕉️ Omni-Indic Translation Suite — GPU Backend

This notebook runs the **entire FastAPI backend** on a Colab GPU.
Your local Next.js frontend connects to it via ngrok.

**Runtime → Change runtime type → T4 GPU**

---

## 1. Install Dependencies

In [1]:
!pip install -q fastapi uvicorn[standard] python-multipart \
    torch transformers sentencepiece \
    easyocr PyMuPDF gTTS \
    numpy Pillow opencv-python-headless \
    pyngrok nest-asyncio

zsh:1: no matches found: uvicorn[standard]


## 2. Language Configuration

In [2]:
LANGUAGES = {
    "telugu":    {"flores": "tel_Telu", "gtts": "te", "label": "Telugu",    "script": "తెలుగు"},
    "tamil":     {"flores": "tam_Taml", "gtts": "ta", "label": "Tamil",     "script": "தமிழ்"},
    "kannada":   {"flores": "kan_Knda", "gtts": "kn", "label": "Kannada",   "script": "ಕನ್ನಡ"},
    "malayalam": {"flores": "mal_Mlym", "gtts": "ml", "label": "Malayalam", "script": "മലയാളം"},
    "marathi":   {"flores": "mar_Deva", "gtts": "mr", "label": "Marathi",   "script": "मराठी"},
    "punjabi":   {"flores": "pan_Guru", "gtts": "pa", "label": "Punjabi",   "script": "ਪੰਜਾਬੀ"},
    "hindi":     {"flores": "hin_Deva", "gtts": "hi", "label": "Hindi",     "script": "हिन्दी"},
}

SOURCE_LANG_FLORES = "eng_Latn"

def get_language(key): return LANGUAGES.get(key.lower())
def get_flores_code(key):
    lang = get_language(key)
    return lang["flores"] if lang else None
def get_gtts_code(key):
    lang = get_language(key)
    return lang["gtts"] if lang else None

print(f"✅ {len(LANGUAGES)} languages configured")

✅ 7 languages configured


## 3. Load Translation Model (NLLB-200 on GPU)

In [3]:
import torch
import re
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_NAME = "facebook/nllb-200-distilled-600M"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Device: {DEVICE}")

print(f"📥 Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
).to(DEVICE)
model.eval()
print(f"✅ Model loaded on {DEVICE}")


def split_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if s.strip()]


def translate(text, target_lang, max_length=256, num_beams=5):
    """Translate English text to an Indian language using NLLB-200."""
    flores_code = get_flores_code(target_lang)
    if not flores_code:
        raise ValueError(f"Unsupported language: {target_lang}")

    tokenizer.src_lang = SOURCE_LANG_FLORES
    sentences = split_sentences(text)
    if not sentences:
        return {"translated_text": "", "confidence": 0.0}

    translated_parts = []
    confidence_scores = []

    for sentence in sentences:
        inputs = tokenizer(
            sentence, return_tensors="pt",
            truncation=True, max_length=512, padding=True
        ).to(DEVICE)

        forced_bos_token_id = tokenizer.convert_tokens_to_ids(flores_code)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                forced_bos_token_id=forced_bos_token_id,
                max_new_tokens=max_length,
                num_beams=num_beams,
                num_return_sequences=1,
                output_scores=True,
                return_dict_in_generate=True,
            )

        translated = tokenizer.batch_decode(output.sequences, skip_special_tokens=True)[0]
        translated_parts.append(translated)

        # Confidence from softmax
        try:
            probs = []
            for score in output.scores:
                softmax = torch.softmax(score, dim=-1)
                max_prob = softmax.max(dim=-1).values.mean().item()
                probs.append(max_prob)
            confidence_scores.append(sum(probs) / len(probs) if probs else 0.0)
        except:
            confidence_scores.append(0.0)

    full_translation = " ".join(translated_parts)
    avg_confidence = sum(confidence_scores) / len(confidence_scores) if confidence_scores else 0.0

    return {
        "translated_text": full_translation,
        "confidence": round(avg_confidence, 4),
    }


# Quick test
result = translate("Hello, how are you?", "hindi")
print(f"🔤 Test: 'Hello, how are you?' → {result['translated_text']}")
print(f"📊 Confidence: {result['confidence']}")

/Users/deekshith/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/deekshith/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🖥️ Device: cpu
📥 Loading facebook/nllb-200-distilled-600M...


`torch_dtype` is deprecated! Use `dtype` instead!


✅ Model loaded on cpu
🔤 Test: 'Hello, how are you?' → हैलो, आप कैसे हैं?
📊 Confidence: 0.6809


## 4. Load OCR Engine

In [4]:
import easyocr
import numpy as np
import cv2

print("📥 Loading EasyOCR...")
ocr_reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
print("✅ EasyOCR loaded")


def extract_text_from_image(image_bytes):
    """Extract text from image bytes using EasyOCR."""
    nparr = np.frombuffer(image_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    results = ocr_reader.readtext(img, detail=0)
    return "\n".join(results).strip()


def extract_text_from_pdf(pdf_bytes):
    """Extract text from PDF bytes using PyMuPDF."""
    import fitz
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    text_parts = []
    for page in doc:
        page_text = page.get_text("text")
        if page_text.strip():
            text_parts.append(page_text.strip())
    doc.close()
    return "\n\n".join(text_parts)

Using CPU. Note: This module is much faster with a GPU.


📥 Loading EasyOCR...
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete✅ EasyOCR loaded


## 5. Build FastAPI Server

In [5]:
import io
from typing import Literal, Optional

from fastapi import FastAPI, File, Form, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel

app = FastAPI(title="Omni-Indic Translation Suite")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Allow all origins for Colab
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


# ─── Request Models ──────────────────────────────────────────────────

class TextTranslateRequest(BaseModel):
    text: str
    target_lang: str
    mode: Literal["translate", "transliterate"] = "translate"

class TTSRequest(BaseModel):
    text: str
    lang: str


# ─── Health & Config ──────────────────────────────────────────────────

@app.get("/")
async def root():
    return {"service": "Omni-Indic Translation Suite", "status": "running", "device": DEVICE}

@app.get("/api/languages")
async def get_languages():
    return {"languages": [
        {"key": k, "label": v["label"], "script": v["script"]}
        for k, v in LANGUAGES.items()
    ]}

@app.get("/api/status")
async def get_status():
    return {
        "translation_engine": True,
        "transliteration_engine": False,
        "ocr_processor": True,
        "pdf_processor": True,
        "device": DEVICE,
    }


# ─── Text Translation ────────────────────────────────────────────────

@app.post("/api/translate-text")
async def translate_text_endpoint(body: TextTranslateRequest):
    text = body.text.strip()
    if not text:
        raise HTTPException(400, "Text cannot be empty.")

    target = body.target_lang.lower()

    if body.mode == "transliterate":
        # Transliteration not available on Colab (fairseq build issue)
        # Return the original text with a note
        return {
            "original_text": text,
            "translated_text": text,
            "target_language": target,
            "mode": "transliterate",
            "confidence": 0.0,
            "extracted_text": None,
        }

    try:
        result = translate(text, target)
        return {
            "original_text": text,
            "translated_text": result["translated_text"],
            "target_language": target,
            "mode": "translate",
            "confidence": result["confidence"],
            "extracted_text": None,
        }
    except ValueError as e:
        raise HTTPException(400, str(e))
    except Exception as e:
        raise HTTPException(500, f"Translation failed: {e}")


# ─── Document Translation ────────────────────────────────────────────

IMAGE_TYPES = {"image/jpeg", "image/png", "image/webp", "image/bmp", "image/tiff"}
PDF_TYPES = {"application/pdf"}

@app.post("/api/translate-doc")
async def translate_doc_endpoint(
    file: UploadFile = File(...),
    target_lang: str = Form(...),
    mode: str = Form("translate"),
):
    content_type = file.content_type or ""
    if content_type not in IMAGE_TYPES | PDF_TYPES:
        raise HTTPException(400, f"Unsupported file type: {content_type}")

    file_bytes = await file.read()
    if len(file_bytes) > 10 * 1024 * 1024:
        raise HTTPException(413, "File too large. Max 10 MB.")

    target = target_lang.lower()

    # Extract text
    if content_type in IMAGE_TYPES:
        extracted = extract_text_from_image(file_bytes)
    else:
        extracted = extract_text_from_pdf(file_bytes)

    if not extracted.strip():
        raise HTTPException(422, "No text could be extracted from the file.")

    # Translate
    result = translate(extracted, target)

    return {
        "original_text": extracted,
        "translated_text": result["translated_text"],
        "target_language": target,
        "mode": mode,
        "confidence": result["confidence"],
        "extracted_text": extracted,
    }


# ─── TTS ─────────────────────────────────────────────────────────────

@app.post("/api/generate-tts")
async def generate_tts_endpoint(body: TTSRequest):
    text = body.text.strip()
    if not text:
        raise HTTPException(400, "Text cannot be empty.")

    gtts_code = get_gtts_code(body.lang.lower())
    if not gtts_code:
        raise HTTPException(400, f"Unsupported TTS language: {body.lang}")

    from gtts import gTTS
    tts = gTTS(text=text, lang=gtts_code, slow=False)
    audio_buffer = io.BytesIO()
    tts.write_to_fp(audio_buffer)
    audio_buffer.seek(0)

    return StreamingResponse(
        audio_buffer,
        media_type="audio/mpeg",
        headers={"Content-Disposition": f'attachment; filename="tts_{body.lang}.mp3"'},
    )


print("✅ FastAPI app built with all endpoints")

✅ FastAPI app built with all endpoints


## 6. Start Server with ngrok

1. Get a **free ngrok auth token** at https://dashboard.ngrok.com/signup
2. Paste it below
3. Run the cell — it will print a **public URL**
4. Set that URL as `NEXT_PUBLIC_API_URL` in your frontend `.env.local`

In [6]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn

# ──────────────────────────────────────────────────────────────────────
# 👇 PASTE YOUR NGROK AUTH TOKEN HERE
NGROK_AUTH_TOKEN = ""  # e.g. "2abc123def456..."
# ──────────────────────────────────────────────────────────────────────

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Allow nested event loops in Colab
nest_asyncio.apply()

# Open ngrok tunnel
public_url = ngrok.connect(8000)
print("═" * 60)
print(f"  🌐 PUBLIC URL: {public_url}")
print("═" * 60)
print()
print("📋 Copy this into your frontend/.env.local:")
print(f"   NEXT_PUBLIC_API_URL={public_url}")
print()
print("Then restart your frontend: npm run dev")
print("═" * 60)

# Start the server (blocking)
uvicorn.run(app, host="0.0.0.0", port=8000)

t=2026-05-13T11:43:10+0530 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:  authentication failed: Usage of ngrok requires a verified account and authtoken.
ERROR:  
ERROR:  Sign up for an account: https://dashboard.ngrok.com/signup
ERROR:  Install your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken
ERROR:  
ERROR:  ERR_NGROK_4018
ERROR:  https://ngrok.com/docs/errors/err_ngrok_4018
ERROR:  
t=2026-05-13T11:43:10+0530 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n

PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

---

## 📌 Quick Reference

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/api/translate-text` | POST | Translate text |
| `/api/translate-doc` | POST | Upload image/PDF for OCR + translation |
| `/api/generate-tts` | POST | Generate TTS audio (MP3) |
| `/api/languages` | GET | List supported languages |
| `/api/status` | GET | Engine status |

### Test with curl:
```bash
curl -X POST <NGROK_URL>/api/translate-text \
  -H "Content-Type: application/json" \
  -d '{"text": "Hello world", "target_lang": "telugu", "mode": "translate"}'
```